----
<div style="display: flex; align-items: center;">
    <img src="https://developers.lseg.com/content/dam/devportal/icons/logo/lseg-logo.svg" width="20%" style="vertical-align: top;">
    <h1 style="margin-left: 20px;">Data Library for Python</h1>
</div>

----

## Discovery layer - Search
This notebook covers the public APIs `search`, `Views`, `search_templates`, `SearchPropertyExplorer` and `PropertyType` exported by `lseg.data.discovery`.
**Prerequisites:** a configured session entitled to Search and Search metadata. Configure credentials in `lseg-data.config.json`, not in this notebook. Results, available options and classifications change over time; service and entitlement errors are not suppressed. No server-side resources are created.

#### Learn more
Visit the [LSEG Developer Portal](https://developers.lseg.com/en) for learning materials.

#### Getting Help and Support
For API questions, visit the [LSEG Developer Community](https://community.developers.lseg.com/).

----

## Set the configuration file location
Run this notebook from its own directory. Discovery notebooks are one directory closer to the project root than the Content examples, so the configuration path is `../../Configuration`.

In [ ]:
import os

os.environ["LD_LIB_CONFIG_PATH"] = os.path.join("..", "..", "Configuration")

## Some Imports to start with

In [ ]:
import lseg.data as ld
from lseg.data.discovery import (
    search, Views, search_templates, SearchPropertyExplorer, PropertyType,
)
from lseg.data.errors import LDError

## Open the data session
`open_session()` uses the configured default session. Select another configured session explicitly if required by your entitlements.

In [ ]:
ld.open_session()

## Free-text search
`search` returns a pandas DataFrame directly, unlike Content layer definitions that return a response object. The default view is `Views.SEARCH_ALL`, and the default result limit (`top`) is 10. The test searches for Microsoft and inspects `BusinessEntity`.

In [ ]:
results = search("Microsoft")
results

## Structured search and boosting
`Views` selects the searchable dataset. `filter` restricts matches, while `boost` raises the ranking of matching documents without excluding others. `select` is a comma-separated property list. This example uses the refinery criteria from the test.

In [ ]:
refineries = search(
    view=Views.PHYSICAL_ASSETS,
    filter="RCSAssetTypeLeaf eq 'oil refinery' and RCSRegionLeaf eq 'Venezuela'",
    boost="PlantStatus ne 'Normal Operation'",
    select="DocumentTitle,BusinessEntity",
)
refineries

## Built-in search templates
`search_templates` exposes embedded templates as attributes. The test uses `UnderlyingRICToOption` to find options on `VOD.L`. It returns a DataFrame, which can legitimately be empty if no options match.

In [ ]:
options = search_templates.UnderlyingRICToOption.search(ric="VOD.L")
options

### User-configured templates
The test also calls `search_templates["Equity"].search(exchange_name="Frankfurt Stock Exchange", navigators="RCSAssetCategoryLeaf(buckets:3, sub:ExchangeName(buckets:2))")` after loading a dedicated test configuration.

`Equity` is not a built-in template: it must be defined under `search.templates` in your library configuration, with an `exchange_name` parameter. The cell below lists configured names and runs that example only when the template is present; otherwise it explicitly reports the missing prerequisite. It does not modify your configuration.

In [ ]:
configured_templates = search_templates.keys()
print("Configured templates:", configured_templates)
if "Equity" in configured_templates:
    display(search_templates["Equity"].search(
        exchange_name="Frankfurt Stock Exchange",
        navigators="RCSAssetCategoryLeaf(buckets:3, sub:ExchangeName(buckets:2))",
    ))
else:
    print("Equity example not run: configure search.templates.Equity with exchange_name first.")

## Explore available properties and their types
`SearchPropertyExplorer.get_properties_for` combines a search sample with metadata for the selected view. Its response exposes `hits_count`, `df`, `properties` and `navigators`. Unlike `search`, it is not itself a DataFrame.

This example is based on the explorer's public documentation. `PropertyType` has `String`, `Double`, `Date`, `Boolean` and `Integer` members. The explorer's `get_by_name`, `get_by_type`, `get_by_value` and `get_navigable` methods return objects with `df` and `properties` attributes. They help identify fields suitable for filters and selections.

In [ ]:
exploration = SearchPropertyExplorer.get_properties_for(
    view=Views.GOV_CORP_INSTRUMENTS,
    query="santander bonds",
    filter="IsPerpetualSecurity ne true and IsActive eq true and not(AssetStatus in ('MAT' 'DC'))",
)
print("Matching documents:", exploration.hits_count)
display(exploration.df.head(20))
display(exploration.get_by_name("active").df)
display(exploration.get_by_type(PropertyType.String).df.head(10))
display(exploration.get_navigable().df.head(10))

## Inspect an invalid-filter error
The test submits `Invalid Filters` and expects an HTTP 400 `LDError`. This intentionally invalid request demonstrates error handling. Only the expected error is handled; other failures propagate.

In [ ]:
try:
    search(view=Views.PHYSICAL_ASSETS, filter="Invalid Filters")
except LDError as error:
    if error.code != 400:
        raise
    print(f"Expected invalid-filter error: {error}")
else:
    raise RuntimeError("The invalid filter was unexpectedly accepted")

## Close the session
Run this cell even if an earlier request fails.

In [ ]:
ld.close_session()